In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [48]:

clean_df = clean_df.drop(columns=["Ticket"])
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    891 non-null    int64  
 1   Pclass      891 non-null    int64  
 2   Sex         891 non-null    int64  
 3   Age         891 non-null    float64
 4   SibSp       891 non-null    int64  
 5   Parch       891 non-null    int64  
 6   Fare        891 non-null    float64
 7   Embarked_C  891 non-null    float64
 8   Embarked_Q  891 non-null    float64
 9   Embarked_S  891 non-null    float64
dtypes: float64(5), int64(5)
memory usage: 69.7 KB


In [31]:
clean_df["Age"] = clean_df.groupby(["Pclass", "Sex"])["Age"].transform(
    lambda x: x.fillna(x.median())
)


In [49]:
clean_df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
0,0,3,1,22.0,1,0,7.2500,0.0,0.0,1.0
1,1,1,0,38.0,1,0,71.2833,1.0,0.0,0.0
2,1,3,0,26.0,0,0,7.9250,0.0,0.0,1.0
3,1,1,0,35.0,1,0,53.1000,0.0,0.0,1.0
4,0,3,1,35.0,0,0,8.0500,0.0,0.0,1.0


In [43]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit + transform into numpy array
embarked_encoded = ohe.fit_transform(clean_df[["Embarked"]])

# Convert to DataFrame with proper column names
embarked_df = pd.DataFrame(
    embarked_encoded,
    columns=ohe.get_feature_names_out(["Embarked"]),
    index=clean_df.index
)

# Concatenate back to original df (drop old column if you want)
clean_df = pd.concat([clean_df.drop(columns=["Embarked"]), embarked_df], axis=1)


In [63]:
X = clean_df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked_C", "Embarked_Q", "Embarked_S"]]
Y = clean_df["Survived"]

In [65]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, Y_train)

LogisticRegression(max_iter=1000)

In [68]:
train_score = model.score(X_train, Y_train)
test_score = model.score(X_test, Y_test)

print("Train accuracy:", train_score)
print("Test accuracy:", test_score)


Train accuracy: 0.8075842696629213
Test accuracy: 0.8156424581005587


In [69]:
import joblib

joblib.dump(model, 'titanic_LR_model.pkl')


['titanic_LR_model.pkl']

In [74]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [75]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, Y_train)

KNeighborsClassifier()

In [77]:
train_score = knn_model.score(X_train_scaled, Y_train)
test_score = knn_model.score(X_test_scaled, Y_test)

print("Train accuracy:", train_score)
print("Test accuracy:", test_score)


Train accuracy: 0.8567415730337079
Test accuracy: 0.7988826815642458


In [78]:
import joblib

joblib.dump(knn_model, 'titanic_KNN_model.pkl')

['titanic_KNN_model.pkl']

In [79]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train, Y_train)


GaussianNB()

In [80]:
train_score = nb_model.score(X_train, Y_train)
test_score = nb_model.score(X_test, Y_test)

print("Train accuracy:", train_score)
print("Test accuracy:", test_score)


Train accuracy: 0.8033707865168539
Test accuracy: 0.7597765363128491


In [81]:
import joblib

joblib.dump(nb_model, 'titanic_NB_model.pkl')

['titanic_NB_model.pkl']

In [82]:
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
692,3,1,25.0,0,0,56.4958,0.0,0.0,1.0
481,2,1,30.0,0,0,0.0000,0.0,0.0,1.0
527,1,1,40.0,0,0,221.7792,0.0,0.0,1.0
855,3,0,18.0,0,1,9.3500,0.0,0.0,1.0
801,2,0,31.0,1,1,26.2500,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...
359,3,0,21.5,0,0,7.8792,0.0,1.0,0.0
258,1,0,35.0,0,0,512.3292,1.0,0.0,0.0
736,3,0,48.0,1,3,34.3750,0.0,0.0,1.0
462,1,1,47.0,0,0,38.5000,0.0,0.0,1.0


In [85]:
from sklearn.svm import SVC

model_svm = SVC(kernel='rbf')
model_svm.fit(X_train_scaled, Y_train)

SVC()

In [86]:
model_svm.predict(X_test_scaled)

array([0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0,
       0, 1, 0])

In [87]:
train_score = model_svm.score(X_train_scaled, Y_train)
test_score = model_svm.score(X_test_scaled, Y_test)

print("Train accuracy:", train_score)
print("Test accuracy:", test_score)


Train accuracy: 0.8455056179775281
Test accuracy: 0.8100558659217877


In [88]:
import joblib

joblib.dump(model_svm, 'titanic_SVM_model.pkl')

['titanic_SVM_model.pkl']